# PostFab BGE-M3 임베딩 파인튜닝

**목표**: BAAI/bge-m3 모델을 반도체 후공정 도메인에 파인튜닝

**데이터**: 54개 corpus 문서 × 4 Q&A = 216쌍 (Train 172 / Test 44)

**예상 소요 시간**: T4 GPU 기준 약 15~20분

## 0. 파일 확인

`train.json`, `test.json`, `ir_eval.json`이 노트북과 같은 폴더에 있으면 그대로 실행하세요.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Drive 내 파일 경로 설정
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/PostFab'
os.chdir(DATA_DIR)

# 파일 확인
for f in ['train.json', 'test.json', 'ir_eval.json']:
    status = '✅' if os.path.exists(f) else '❌ 없음'
    print(f'{f}: {status}')

## 1. 패키지 설치

In [ ]:
!pip install -q -U sentence-transformers datasets
!pip install -q -U huggingface_hub

## 2. GPU 확인

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 - Runtime > Change runtime type > T4 GPU 선택"}')

## 3. Before 평가 (파인튜닝 전 성능 측정)

In [ ]:
import json
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

with open('ir_eval.json', encoding='utf-8') as f:
    ir = json.load(f)

corpus = ir['corpus']
queries = ir['queries']
relevant_docs = {k: set(v) for k, v in ir['relevant_docs'].items()}

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='postfab-test',
    accuracy_at_k=[1, 3, 5],
    ndcg_at_k=[10],
    show_progress_bar=True
)

model_before = SentenceTransformer('BAAI/bge-m3')
results_before = evaluator(model_before)
print('\n=== Before 파인튜닝 ===')
print(f'Accuracy@1:  {results_before["postfab-test_cosine_accuracy@1"]:.4f}')
print(f'Accuracy@3:  {results_before["postfab-test_cosine_accuracy@3"]:.4f}')
print(f'Accuracy@5:  {results_before["postfab-test_cosine_accuracy@5"]:.4f}')
print(f'NDCG@10:     {results_before["postfab-test_cosine_ndcg@10"]:.4f}')

## 4. 파인튜닝

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from datasets import Dataset
import torch, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

with open('train.json', encoding='utf-8') as f:
    train_data = json.load(f)

train_dataset = Dataset.from_list(train_data)

model = SentenceTransformer('BAAI/bge-m3')
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir='postfab-bge-m3',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    fp16=True,
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=evaluator,
)

trainer.train()

## 5. After 평가 (파인튜닝 후 성능 측정)

In [ ]:
results_after = evaluator(model)

print('\n========== 결과 비교 ==========')
metrics = [
    ('Accuracy@1', 'postfab-test_cosine_accuracy@1'),
    ('Accuracy@3', 'postfab-test_cosine_accuracy@3'),
    ('Accuracy@5', 'postfab-test_cosine_accuracy@5'),
    ('NDCG@10',    'postfab-test_cosine_ndcg@10'),
]
for label, key in metrics:
    before = results_before[key]
    after = results_after[key]
    diff = after - before
    arrow = '▲' if diff > 0 else '▼'
    print(f'{label:15s}: {before:.4f} → {after:.4f}  {arrow} {abs(diff):.4f}')

## 6. 모델 저장 및 다운로드

In [ ]:
model.save('postfab-bge-m3-final')
print('모델 저장 완료: postfab-bge-m3-final/')

# zip으로 압축 후 다운로드
!zip -r postfab-bge-m3-final.zip postfab-bge-m3-final/
files.download('postfab-bge-m3-final.zip')